# PA Engine — weights + characteristics, 0CQ single frequency (PLACEHOLDER)

**Status: stub.** The call shape, auth, polling and landing are here and correct against
upstream PAEngine v3 (SDK 4.0.0). What is *not* here is anything workstation-sourced —
PA document path, component names, account paths, column ids. Nothing runs until those
are filled in via the discovery cell.

Companion to `spar_composite_returns_template.ipynb`. That one is **returns-based**
(SPAR, no holdings needed); this one is **holdings-based** (PA), which is what weights
and characteristics require.

## What this pulls

A point-in-time snapshot as of the most recent calendar quarter end — `frequency:
"Single"`, `enddate: "0CQ"`. Not a time series: one observation per holding (or per
group) at one date.

| Tile | `componentdetail` | Grain |
|---|---|---|
| `weights` | `SECURITIES` | one row per holding |
| `characteristics` | `GROUPS` | one row per group (sector, or whatever the component groups by) |

`componentdetail` accepts `GROUPS`, `GROUPSALL`, `TOTALS`, or `SECURITIES`.

## Why this is one multi-port call, unlike the SPAR notebook

`PACalculationParameters.benchmarks` is a **list**; SPAR's `benchmark` is a scalar. That
single difference is what makes a genuine multi-port call possible here — all four
composites and their benchmarks go in one unit per tile, instead of SPAR's one unit per
(tile, strategy, basis).

The other reason it collapses cleanly: this is a **single-date snapshot**, so there is no
per-strategy inception window to preserve. The SPAR notebook can't collapse precisely
because its inception-to-date tiles need a different `startdate` per strategy, and a unit
carries one `dates` object. Here every account shares the same one date.

> ⚠️ **Verify before trusting: how PA pairs a list of accounts with a list of
> benchmarks.** Positional pairing (account[i] vs benchmark[i]) and cross-product (every
> account vs every benchmark) are both plausible readings of the schema, and the docs do
> not say which. If the output doesn't show each composite against its own benchmark, set
> `MULTIPORT = False` to fall back to one unit per strategy, which is unambiguous. Do not
> ship a tearsheet off this until that is confirmed.

## Fee basis does not apply here

Weights and characteristics are holdings attributes — there is no gross/net distinction
to make. The 2x fee-basis fan-out from the SPAR notebook is absent by design.

## Versions

| Package | Version |
|---|---|
| `fds.sdk.PAEngine` | **4.0.0** (upstream latest, 2026-07-21) |
| `fds.sdk.utils` | 3.0.1 |
| `fds.protobuf.stach.extensions` | 1.3.3 |
| `deltalake` | 1.6.2 |

**PAEngine 4.0.0 carries a breaking change beyond the 2026-05-20 Python-wide bump.** Per
upstream `BREAKING.md` (2026-07-21), `PADateParameters.enddate` and `.frequency` both
lost `required: true` and the schema's `required` array was dropped entirely, which
**reshuffles positional arguments** across ~6 operations including the calculation
endpoints and `convertPADatesToAbsoluteFormat`. A new optional `calendar` parameter was
also added.

The migration note is to use named parameters. Every call in this notebook passes
keywords, so it is unaffected — but any older PA code passing dates positionally will
silently bind the wrong values, which is worth auditing before this goes near production.

> ⚠️ The PA SDK vendored under `code/python/PAEngine/v3/` in this repo is **2.2.2** and
> predates all of the above. Verify against upstream `main`.

Attach libraries to a **Fabric Environment**, not `%pip` — inline installs are disabled
by default in pipeline runs and unsupported in reference runs. Interactive first run only:
```
%pip install fds.sdk.PAEngine==4.0.0 fds.sdk.utils==3.0.1 \
             fds.protobuf.stach.extensions==1.3.3 deltalake==1.6.2
```

In [ ]:
# === Cell 1: credentials ===================================================
# Same HBCM_Config notebook as the SPAR pipeline — one place for the FactSet key.
%run HBCM_Config

In [ ]:
# === Cell 2: imports + API client ==========================================
import json, time, datetime as dt
import pandas as pd

import fds.sdk.PAEngine
from fds.sdk.PAEngine.api import (
    pa_calculations_api,
    components_api,
    accounts_api,
    columns_api,
    groups_api,
    frequencies_api,
    dates_api,          # PA HAS a DatesApi — SPAR does not. See Cell 3.
)
from fds.sdk.PAEngine.models import (
    PACalculationParametersRoot,
    PACalculationParameters,
    PAIdentifier,
    PADateParameters,
    CalculationMeta,
)
from urllib3 import Retry

SDK_VERSION = fds.sdk.PAEngine.__version__
assert int(SDK_VERSION.split(".")[0]) >= 4, (
    f"fds.sdk.PAEngine {SDK_VERSION} found; this notebook targets >=4.0.0 "
    "(PADateParameters changed shape). Check the bound Fabric Environment."
)
print("PAEngine SDK", SDK_VERSION)

configuration = fds.sdk.PAEngine.Configuration(
    username=FACTSET_USER,
    password=FACTSET_APIKEY,
)
configuration.retries = Retry(
    total=3, status_forcelist=[500, 502, 503, 504], backoff_factor=2,
    allowed_methods=frozenset(["GET", "POST"]),
)

api_client = fds.sdk.PAEngine.ApiClient(configuration)
calc_api = pa_calculations_api.PACalculationsApi(api_client)
comp_api = components_api.ComponentsApi(api_client)

In [ ]:
# === Cell 3: THE CONFIG BLOCK ==============================================

CURRENCY = "USD"

# Point-in-time snapshot: one date, no series. Single frequency means startdate is
# irrelevant to the result, but PA still wants a coherent window, so it is set to the
# same date.
AS_OF_RELATIVE = "0CQ"
FREQUENCY = "Single"
USE_ABSOLUTE_AS_OF = False

def _prior_quarter_end(today=None):
    d = today or dt.date.today()
    qe = dt.date(d.year, ((d.month - 1) // 3) * 3 + 1, 1) - dt.timedelta(days=1)
    return qe.strftime("%Y%m%d")

AS_OF_ABS = _prior_quarter_end()
AS_OF = AS_OF_ABS if USE_ABSOLUTE_AS_OF else AS_OF_RELATIVE

# Unlike SPAR, PA exposes DatesApi.convert_pa_dates_to_absolute_format(), so 0CQ CAN be
# resolved server-side before submitting and checked against AS_OF_ABS. Cell 4 does it.
# This is the verification the SPAR notebook cannot perform.

# --- accounts --------------------------------------------------------------
# Same four composites as the SPAR notebook, but PA needs the HOLDINGS account path, not
# the returns ACCT. holdingsmode: B&H, TBR, OMS, EXT or VLT.
STRATEGIES = {
    "LC":   {"label": "Large Cap",           "acct": "<TODO path.ACCT>",
             "holdingsmode": "B&H", "benchmark": "<TODO R1000>"},
    "SMID": {"label": "SMID",                "acct": "<TODO path.ACCT>",
             "holdingsmode": "B&H", "benchmark": "<TODO>"},
    "LCS":  {"label": "Large Cap Select",    "acct": "<TODO path.ACCT>",
             "holdingsmode": "B&H", "benchmark": "<TODO R1000>"},   # shares LC's bench
    "CONC": {"label": "Concentrated Equity", "acct": "<TODO path.ACCT>",
             "holdingsmode": "B&H", "benchmark": "<TODO>"},
}

# --- tiles -----------------------------------------------------------------
# Resolved by name each run, same rationale as the SPAR notebook: a re-saved component
# can change id, and a stale id 400s with no useful message.
PA_DOCUMENT = "<TODO Client:/PA3/HBCM>"

TILES = {
    "weights": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        "componentdetail": "SECURITIES",   # one row per holding
    },
    "characteristics": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        "componentdetail": "GROUPS",       # one row per group
    },
}

# True  -> one unit per tile, all four accounts + benchmarks in it (the multi-port call).
# False -> one unit per (tile, strategy). Unambiguous pairing; use this until the
#          account<->benchmark pairing question in the header is settled.
MULTIPORT = True

# --- OneLake target --------------------------------------------------------
WORKSPACE_ID = "1b9fac18-9d75-4437-ab6c-b6ba44ff46a8"   # HBCM - Production
LAKEHOUSE_ID = "7cdf13b1-4586-4a02-b8ff-72fcf6db1277"   # hbcm_datahub
ONELAKE = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}"
TABLE_PATH = f"{ONELAKE}/Tables/factset/pa_weights_characteristics"
RAW_DIR = f"{ONELAKE}/Files/raw/pa"

print(f"as-of sent: {AS_OF!r}  frequency: {FREQUENCY}  "
      f"asof_date written: {AS_OF_ABS}  multiport: {MULTIPORT}")

In [ ]:
# === Cell 4: resolve dates + component ids, every run =====================

# (a) Resolve 0CQ server-side and confirm it agrees with the locally computed quarter
#     end. PA can do this; SPAR cannot. If they disagree, the asof_date label is wrong.
d_api = dates_api.DatesApi(api_client)
try:
    resolved_dates = d_api.convert_pa_dates_to_absolute_format(
        startdate=AS_OF, enddate=AS_OF, componentid=None, account=None,
    )
    print("PA resolved dates:", resolved_dates)
    print(f"locally computed quarter end: {AS_OF_ABS}")
except Exception as e:
    # Signature takes optional componentid/account and shifted in 4.0.0 — if this errors,
    # check the current arg list rather than assuming 0CQ is wrong.
    print(f"date conversion unavailable ({e!r}); relying on AS_OF_ABS for labelling")

# (b) component name -> live id
def _field(obj, name):
    if hasattr(obj, name):
        return getattr(obj, name)
    try:
        return obj.get(name)
    except AttributeError:
        return None

def resolve_component_ids(document=PA_DOCUMENT):
    summary = comp_api.get_pa_components(document=document)
    by_name = {}
    for cid, meta in (summary.data or {}).items():
        by_name.setdefault(_field(meta, "name"), []).append(cid)

    resolved, drift, missing = {}, [], []
    for tile, cfg in TILES.items():
        hits = by_name.get(cfg["component_name"], [])
        if len(hits) != 1:
            missing.append((tile, cfg["component_name"], len(hits)))
            continue
        resolved[tile] = hits[0]
        if cfg.get("pinned_componentid") and cfg["pinned_componentid"] != hits[0]:
            drift.append((tile, cfg["component_name"], cfg["pinned_componentid"], hits[0]))
    return resolved, drift, missing, by_name

RESOLVED_COMPONENTS, COMPONENT_DRIFT, COMPONENT_MISSING, COMPONENTS_BY_NAME = \
    resolve_component_ids()

for tile, cid in RESOLVED_COMPONENTS.items():
    print(f"{tile:<18} {cid}  ({TILES[tile]['component_name']})")

if COMPONENT_DRIFT:
    print("\n*** COMPONENT ID DRIFT — the component was re-saved. Confirm its columns")
    print("*** still match, then update pinned_componentid:")
    for tile, name, was, now in COMPONENT_DRIFT:
        print(f"    {tile}: {name!r}  {was} -> {now}")

if COMPONENT_MISSING:
    print("\nUnresolved tiles:", COMPONENT_MISSING)
    print("Available component names:")
    for name, cids in sorted(COMPONENTS_BY_NAME.items(), key=lambda kv: str(kv[0])):
        print(f"    {name!r}: {cids}")

assert not COMPONENT_MISSING, "every tile needs exactly one matching component name"

In [ ]:
# === Cell 4b: DISCOVERY — run once interactively ==========================
# Fills the TODOs in Cell 3. Not part of the scheduled path.

a_api = accounts_api.AccountsApi(api_client)
col_api = columns_api.ColumnsApi(api_client)
grp_api = groups_api.GroupsApi(api_client)
frq_api = frequencies_api.FrequenciesApi(api_client)

# Account paths -> STRATEGIES[...]["acct"]
#   print(a_api.get_accounts(path="Client:/"))

# Available columns -> confirm the weights / characteristics components expose what you
# expect. PACalculationParameters.columns can override the component's saved columns.
#   print(col_api.get_pa_columns(name="weight", category="", directory=""))

# Groupings -> PACalculationParameters.groups overrides the document's grouping
#   print(grp_api.get_pa_groups())

# Frequencies -> confirm "Single" is spelled as PA expects
#   print(frq_api.get_pa_frequencies())
print("uncomment the lookup you need")

In [ ]:
# === Cell 5: build units ==================================================
# PA takes accounts AND benchmarks as lists, so MULTIPORT puts all four composites in one
# unit per tile. Safe here only because this is a single-date snapshot — every account
# shares one `dates` object, which is exactly the constraint that blocks collapsing in the
# SPAR notebook's inception-to-date tiles.

def pa_dates() -> PADateParameters:
    # Keyword args throughout: 4.0.0 dropped `required` on enddate/frequency, which
    # reshuffled positional arguments across the PA calculation endpoints.
    return PADateParameters(startdate=AS_OF, enddate=AS_OF, frequency=FREQUENCY)

def build_multiport_unit(tile_name: str, tile_cfg: dict) -> PACalculationParameters:
    return PACalculationParameters(
        componentid=RESOLVED_COMPONENTS[tile_name],
        accounts=[PAIdentifier(id=s["acct"], holdingsmode=s["holdingsmode"])
                  for s in STRATEGIES.values()],
        # De-duplicated: LC and LCS share a benchmark, so it appears once.
        benchmarks=[PAIdentifier(id=b) for b in
                    dict.fromkeys(s["benchmark"] for s in STRATEGIES.values())],
        dates=pa_dates(),
        currencyisocode=CURRENCY,
        componentdetail=tile_cfg["componentdetail"],
    )

def build_single_unit(tile_name: str, tile_cfg: dict, code: str) -> PACalculationParameters:
    s = STRATEGIES[code]
    return PACalculationParameters(
        componentid=RESOLVED_COMPONENTS[tile_name],
        accounts=[PAIdentifier(id=s["acct"], holdingsmode=s["holdingsmode"])],
        benchmarks=[PAIdentifier(id=s["benchmark"])],
        dates=pa_dates(),
        currencyisocode=CURRENCY,
        componentdetail=tile_cfg["componentdetail"],
    )

UNIT_KEYS, units = {}, {}
for tile_name, tile_cfg in TILES.items():
    if MULTIPORT:
        key = tile_name
        units[key] = build_multiport_unit(tile_name, tile_cfg)
        UNIT_KEYS[key] = (tile_name, None)      # strategy comes from the output rows
    else:
        for code in STRATEGIES:
            key = f"{tile_name}__{code}"
            units[key] = build_single_unit(tile_name, tile_cfg, code)
            UNIT_KEYS[key] = (tile_name, code)

params_root = PACalculationParametersRoot(
    data=units,
    meta=CalculationMeta(
        contentorganization="SimplifiedRow",
        stach_content_organization="SimplifiedRow",
        contenttype="Json",
        format="JsonStach",
    ),
)
print(f"built {len(units)} units: {list(units)}")

In [ ]:
# === Cell 6: submit + poll ================================================
# Identical semantics to SPAR: 200 sync, 201 ready, 202 poll. Multi-unit always 202.

def run_pa(params_root, deadline=10, poll_interval=3, timeout=900):
    wrapper = calc_api.post_and_calculate(
        x_fact_set_api_long_running_deadline=deadline,
        pa_calculation_parameters_root=params_root,
    )
    code = wrapper.get_status_code()
    if code == 200:
        status_root = wrapper.get_response_200()
    elif code == 201:
        status_root = wrapper.get_response_201()
    elif code == 202:
        status_root = wrapper.get_response_202()
        calc_id = status_root.data.calculationid
        deadline_at = time.time() + timeout
        while True:
            if time.time() > deadline_at:
                calc_api.cancel_calculation_by_id(id=calc_id)
                raise TimeoutError(f"calc {calc_id} exceeded {timeout}s (cancelled)")
            poll = calc_api.get_calculation_status_by_id(id=calc_id)
            if poll.get_status_code() == 200:
                status_root = poll.get_response_200()
                break
            if poll.get_status_code() != 202:
                raise RuntimeError(f"unexpected poll status {poll.get_status_code()}")
            time.sleep(poll_interval)
    else:
        raise RuntimeError(f"unexpected submit status {code}")

    calc_id = status_root.data.calculationid
    out = []
    for unit_id, unit_status in (status_root.data.units or {}).items():
        st = getattr(unit_status, "status", None)
        if st != "Success":
            out.append((unit_id, None, st))
            continue
        out.append((unit_id,
                    calc_api.get_calculation_unit_result_by_id(id=calc_id, unit_id=unit_id),
                    st))
    return calc_id, out

calc_id, results = run_pa(params_root)
failed = [(u, s) for u, r, s in results if r is None]
print(f"calc={calc_id} ok={len(results) - len(failed)} failed={len(failed)}")
for u, s in failed:
    print(f"  FAILED {u}: {s}")
assert results and not failed, "resolve failures before writing to the lakehouse"

In [ ]:
# === Cell 7: raw landing + STACH -> DataFrame ==============================
from fds.protobuf.stach.extensions.StachExtensionFactory import StachExtensionFactory
from fds.protobuf.stach.extensions.StachVersion import StachVersion

asof_tag = AS_OF_ABS
for unit_id, res, _ in results:
    notebookutils.fs.put(f"{RAW_DIR}/asof={asof_tag}/{unit_id}.json",
                         json.dumps(res.to_dict(), default=str), True)
print(f"landed {len(results)} raw payloads")

def stach_to_dataframes(api_response):
    ext = StachExtensionFactory.get_stach_extension(StachVersion.V2)
    return [pd.DataFrame(t.data, columns=t.columns)
            for t in ext.convert(json.dumps(api_response.to_dict(), default=str))]

frames = []
for unit_id, res, _ in results:
    tile_name, code = UNIT_KEYS[unit_id]
    for i, df in enumerate(stach_to_dataframes(res)):
        df = df.copy()
        df.insert(0, "asof_date", asof_tag)
        df.insert(1, "tile", tile_name)
        # In MULTIPORT mode the strategy is NOT known from the unit key — it has to come
        # from a portfolio/account column in the output. Map it after inspecting columns.
        df.insert(2, "strategy_code", code)
        df.insert(3, "componentid", RESOLVED_COMPONENTS[tile_name])
        df.insert(4, "componentdetail", TILES[tile_name]["componentdetail"])
        df.insert(5, "table_ix", i)
        frames.append(df)

tidy = pd.concat(frames, ignore_index=True)
tidy.columns = [str(c).strip().replace(" ", "_").lower() for c in tidy.columns]
print(tidy.shape)
print("\ncolumns:", list(tidy.columns))
if MULTIPORT:
    print("\nMULTIPORT: find the account/portfolio column below and map it to a strategy")
    print("code before writing, otherwise strategy_code stays null for every row.")
display(tidy.head(30))

In [ ]:
# === Cell 8: write to hbcm_datahub =========================================
# Left commented until strategy_code is genuinely populated. Writing a table whose
# strategy column is null on every row is worse than not writing it — it looks loaded.

# from deltalake import DeltaTable, write_deltalake
#
# assert tidy["strategy_code"].notna().all(), \
#     "map the output's account column to strategy_code first (see Cell 7)"
#
# try:
#     DeltaTable(TABLE_PATH).delete(f"asof_date = '{asof_tag}'")
#     mode = "append"
# except Exception:
#     mode = "overwrite"
#
# write_deltalake(TABLE_PATH, tidy, mode=mode, schema_mode="merge")
# print(f"wrote {len(tidy)} rows to factset.pa_weights_characteristics (mode={mode})")
print("write step disabled — this notebook is a placeholder")

## To finish this notebook

1. Fill the Cell 3 TODOs: `PA_DOCUMENT`, the two `component_name`s, and each strategy's
   holdings `acct` path and benchmark. Cell 4b's lookups find all of them.
2. Confirm `holdingsmode`. `B&H` (buy & hold) is the default guess; `TBR`, `OMS`, `EXT`
   and `VLT` are the alternatives, and the right one depends on how the composites are
   maintained.
3. **Settle the multi-port pairing question.** Run once with `MULTIPORT = True` and check
   whether each composite appears against its own benchmark or against all of them. If
   the latter, use `MULTIPORT = False`.
4. Map the output's account/portfolio column to `strategy_code` (Cell 7), then uncomment
   the write in Cell 8.
5. Decide whether characteristics wants `GROUPS` or `GROUPSALL`, and whether the
   component's saved grouping is right or should be overridden via
   `PACalculationParameters.groups`.

## Open questions worth resolving before this is trusted

- **Account↔benchmark pairing in multi-port.** The blocking one. See above.
- **Does `Single` frequency at `0CQ` return holdings *as of* the quarter end, or the
  quarter's activity?** Matters for whether weights are a snapshot or an average.
- **Weights vs characteristics as one table or two.** They have different grains
  (`SECURITIES` vs `GROUPS`), so one table means a mostly-null wide row set. Two tables
  is probably right; this stub writes one for simplicity and should likely be split.
- **Whether the SPAR and PA snapshots need to reconcile.** They come from different
  engines against different account objects (returns ACCT vs holdings account), so
  agreement is not automatic.

## Sources

Verified against **upstream `FactSet/enterprise-sdk` `main`** (PAEngine v3, SDK 4.0.0) —
`PACalculationParameters.md` (`accounts`/`benchmarks` as lists, `componentdetail`,
`columns`, `groups`), `PADateParameters.md`, `PAIdentifier.md` (`holdingsmode` values),
`PACalculationsApi.md`, and `BREAKING.md` (2026-07-21 `PADateParameters`; 2026-05-20
Python-wide bump).

Not against `code/python/PAEngine/v3/` in this repo, which is pinned at 2.2.2 and
predates the 4.0.0 changes.